# PASO 2 : Feature Engineering

**Objetivo:** Procesar `dataset_clean.csv` y convertirlo en la versión final optimizada para consumo de algoritmos Machine Learning (los modelos).

In [47]:
# Carga de librerías esenciales y DataFrame
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer

# Importamos el dataset con el que vamos a trabajar y mostramos las primeras 3 filas del mismo
df = pd.read_csv('../data/dataset_clean.csv')
display(df.head(3))

,donde_estas_trabajando,dedicacion,tipo_de_contrato,modalidad_de_trabajo,sueldo_dolarizado,contas_con_beneficios_adicionales,estas_buscando_trabajo,tenes_guardias,trabajo_de,seniority,...,bases_de_datos,qa_testing,cantidad_de_personas_en_tu_organizacion,maximo_nivel_de_estudios,estado,carrera,tengo_edad,genero,salario,es_jerarquico
0,Mendoza,Full-Time,Staff (Planta Permanente),Híbrido (Presencial Y Remoto),True,"Capacitaciones Y/O Cursos, Descuento En Gimnas...","No, Pero Escucho Ofertas.",NaN,Developer,Senior,...,"Amazon Dynamodb, Elasticsearch, Google Cloud S...","Junit, Selenium",De 1001 A 2000 Personas,NaN,NaN,NaN,45,Mujer Cis,3000000.0,False
1,Santa Fe,Full-Time,Contractor,100% Remoto,True,Ninguna De Las Anteriores,"No, Estoy Muy Conforme.",NaN,Developer,Senior,...,Microsoft Sql Server,Postman,De 51 A 100 Personas,NaN,NaN,NaN,50,Mujer Cis,5000000.0,False
2,Buenos Aires,Full-Time,Tercerizado (Trabajo A Través De Consultora O ...,100% Remoto,True,"Abono De Internet, Clases De Idiomas, Crédito ...","No, Pero Escucho Ofertas.",No,Developer,Senior,...,Amazon Rds/Aurora,Postman,De 51 A 100 Personas,Universitario,Completo,Ingeniería En Sistemas De Información,36,Mujer Cis,4600000.0,False


In [48]:
# Selección de columnas por tipo
num_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
cat_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]

print("Cantidad de columnas numéricas:", len(num_cols))
print("Numéricas:", num_cols)
print("-" * 100)
print("Cantidad de columnas categóricas:", len(cat_cols))
print("Categóricas:", cat_cols)

Cantidad de columnas numéricas: 7
Numéricas: ['sueldo_dolarizado', 'anos_de_experiencia', 'antiguedad_en_la_empresa_actual', 'anos_en_el_puesto_actual', 'tengo_edad', 'salario', 'es_jerarquico']
----------------------------------------------------------------------------------------------------
Cantidad de columnas categóricas: 19
Categóricas: ['donde_estas_trabajando', 'dedicacion', 'tipo_de_contrato', 'modalidad_de_trabajo', 'contas_con_beneficios_adicionales', 'estas_buscando_trabajo', 'tenes_guardias', 'trabajo_de', 'seniority', 'plataformas_que_utilizas_en_tu_puesto_actual', 'lenguajes_de_programacion_o_tecnologias_que_utilices_en_tu_puesto_actual', 'frameworksherramientas_y_librerias_que_utilices_en_tu_puesto_actual', 'bases_de_datos', 'qa_testing', 'cantidad_de_personas_en_tu_organizacion', 'maximo_nivel_de_estudios', 'estado', 'carrera', 'genero']


In [49]:
# Agrupamos columnas por tipo de variable para facilitar el preprocesamiento y modelado

y = df["salario"].copy()
X = df.copy()

# 1. Numéricas continuas
num_cols = [
    "anos_de_experiencia",
    "antiguedad_en_la_empresa_actual",
    "anos_en_el_puesto_actual",
    "tengo_edad"
]

# 2. Indicadores binarios
bin_cols = [
    "sueldo_dolarizado",
    "es_jerarquico"
]

# 3. Categórica ordinal
ordinal_cols = ["seniority"]

# 4. Columnas con múltiples respuestas
multi_cols = [
    "contas_con_beneficios_adicionales",
    "plataformas_que_utilizas_en_tu_puesto_actual",
    "lenguajes_de_programacion_o_tecnologias_que_utilices_en_tu_puesto_actual",
    "frameworksherramientas_y_librerias_que_utilices_en_tu_puesto_actual",
    "bases_de_datos",
    "qa_testing"
]

# 5. Categóricas nominales
# Tomamos las columnas de texto restantes que no sean ordinales ni multirrespuesta
nom_cols = [
    c for c in X.select_dtypes(include=["object", "category", "string"]).columns
    if c not in ordinal_cols + multi_cols
]

# Verificamos que todas las columnas indicadas existan
for grupo, columnas in {
    "Numéricas": num_cols,
    "Binarias": bin_cols,
    "Ordinales": ordinal_cols,
    "Multirrespuesta": multi_cols,
    "Nominales": nom_cols
}.items():
    faltantes = [c for c in columnas if c not in X.columns]
    if faltantes:
        raise ValueError(f"Columnas faltantes en {grupo}: {faltantes}")

print("Numéricas:", num_cols)
print("Binarias:", bin_cols)
print("Ordinales:", ordinal_cols)
print("Multirrespuesta:", multi_cols)
print("Nominales:", nom_cols)

print("\nDimensiones originales de X:", X.shape)

Numéricas: ['anos_de_experiencia', 'antiguedad_en_la_empresa_actual', 'anos_en_el_puesto_actual', 'tengo_edad']
Binarias: ['sueldo_dolarizado', 'es_jerarquico']
Ordinales: ['seniority']
Multirrespuesta: ['contas_con_beneficios_adicionales', 'plataformas_que_utilizas_en_tu_puesto_actual', 'lenguajes_de_programacion_o_tecnologias_que_utilices_en_tu_puesto_actual', 'frameworksherramientas_y_librerias_que_utilices_en_tu_puesto_actual', 'bases_de_datos', 'qa_testing']
Nominales: ['donde_estas_trabajando', 'dedicacion', 'tipo_de_contrato', 'modalidad_de_trabajo', 'estas_buscando_trabajo', 'tenes_guardias', 'trabajo_de', 'cantidad_de_personas_en_tu_organizacion', 'maximo_nivel_de_estudios', 'estado', 'carrera', 'genero']

Dimensiones originales de X: (4387, 26)


#### Escalado de variables numéricas

In [50]:
# Seleccionamos las variables numéricas
X_num_original = X[num_cols].copy()

# Aplicamos StandardScaler
scaler = StandardScaler()

X_num = pd.DataFrame(
    scaler.fit_transform(X_num_original),
    columns=num_cols,
    index=X.index
)

print("Columnas numéricas originales:", X_num_original.shape[1])
print("Columnas numéricas resultantes:", X_num.shape[1])
print("Columnas nuevas agregadas:", X_num.shape[1] - X_num_original.shape[1])

display(X_num.head())

print("\nMedias después del escalado:")
display(X_num.mean().round(3))

print("\nDesviaciones estándar después del escalado:")
display(X_num.std(ddof=0).round(3))

Columnas numéricas originales: 4
Columnas numéricas resultantes: 4
Columnas nuevas agregadas: 0


,anos_de_experiencia,antiguedad_en_la_empresa_actual,anos_en_el_puesto_actual,tengo_edad
0,0.613263,-0.289608,-0.153412,0.894996
1,1.846085,1.419854,2.189779,1.461241
2,-0.372994,-0.669488,0.888007,-0.124246
3,1.846085,0.280213,0.627652,2.593733
4,-0.866122,-0.859429,-0.934475,-0.916990



Medias después del escalado:


anos_de_experiencia               -0.0
antiguedad_en_la_empresa_actual    0.0
anos_en_el_puesto_actual           0.0
tengo_edad                         0.0
dtype: float64


Desviaciones estándar después del escalado:


anos_de_experiencia                1.0
antiguedad_en_la_empresa_actual    1.0
anos_en_el_puesto_actual           1.0
tengo_edad                         1.0
dtype: float64

#### Codificación de categóricas nominales

In [51]:
# Reemplazamos los valores ausentes por una categoría explícita
X_nom_original = (
    X[nom_cols]
    .astype(str)
    .apply(lambda col: col.str.strip())
)

# One-Hot Encoding
X_nom = pd.get_dummies(
    X_nom_original,
    columns=nom_cols,
    dtype=int
)

print("Columnas nominales originales:", X_nom_original.shape[1])
print("Columnas nominales resultantes:", X_nom.shape[1])
print("Columnas nuevas agregadas:", X_nom.shape[1] - X_nom_original.shape[1])

display(X_nom.head())

Columnas nominales originales: 12
Columnas nominales resultantes: 361
Columnas nuevas agregadas: 349


,donde_estas_trabajando_Buenos Aires,donde_estas_trabajando_Catamarca,donde_estas_trabajando_Chaco,donde_estas_trabajando_Chubut,donde_estas_trabajando_Ciudad Autónoma De Buenos Aires,donde_estas_trabajando_Corrientes,donde_estas_trabajando_Córdoba,donde_estas_trabajando_Entre Ríos,donde_estas_trabajando_Formosa,donde_estas_trabajando_Jujuy,...,carrera_Tecnico Universitario En Programacion,carrera_Telecomunicaciones,carrera_Traducción Literaria Y Técnico-Científica En Inglés,carrera_Traducción/Interpretación Español-Inglés,carrera_Ts Seguridad Laboral,carrera_Técnico Superior En Recursos Humanos,carrera_Técnico Universitario En Redes Y Telecomunicaciones,"carrera_Técnico Universitario En Redes, Lic. En Tecnología Educativa, Prof. En Disciplinas Industriales",genero_Hombre Cis,genero_Mujer Cis
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0


#### Codificación ordinal de `seniority`

In [52]:
# Definimos la jerarquía
seniority_map = {
    "Junior": 0,
    "Semi-Senior": 1,
    "Senior": 2
}

X_ordinal = X[ordinal_cols].copy()

# Estandarizamos el texto de la columna
X_ordinal["seniority"] = (
    X_ordinal["seniority"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convertimos las categorías a números
X_ordinal["seniority"] = X_ordinal["seniority"].map(seniority_map)

# Verificamos si quedaron categorías sin mapear
if X_ordinal["seniority"].isna().any():
    desconocidos = X.loc[
        X_ordinal["seniority"].isna(), "seniority"
    ].value_counts(dropna=False)

    print("Revisar estos valores de seniority:")
    print(desconocidos)

# Contamos cuántos valores fueron convertidos correctamente
valores_codificados = X_ordinal["seniority"].notna().sum()

print("CODIFICACIÓN ORDINAL de 'seniority'")
print("=" * 50)
print("Mapeo aplicado: Junior → 0 | Semi-Senior → 1 | Senior → 2")
print(f"Registros con seniority codificado: {valores_codificados}")
print("Columnas conservadas: 1 (sin generar columnas adicionales)")


CODIFICACIÓN ORDINAL de 'seniority'
Mapeo aplicado: Junior → 0 | Semi-Senior → 1 | Senior → 2
Registros con seniority codificado: 4387
Columnas conservadas: 1 (sin generar columnas adicionales)


#### Codificación de columnas con múltiples respuestas

In [53]:
# Codificación de columnas multirrespuesta
X_multi_parts = {}

# Cantidad mínima de apariciones para crear una columna
MIN_FRECUENCIA = 1

for col in multi_cols:

    # Convertimos cada celda en una lista de respuestas
    listas = (
        X[col]
        .fillna("")
        .astype(str)
        .apply(
            lambda valor: [
                item.strip()
                for item in valor.split(",")
                if item.strip()
            ]
        )
    )

    # Contamos cuántas veces aparece cada opción
    frecuencias = listas.explode().value_counts()

    # Conservamos únicamente las opciones con más de 20 apariciones
    opciones_frecuentes = frecuencias[
        frecuencias > MIN_FRECUENCIA
    ].index.tolist()

    # Filtramos las listas para conservar solo esas opciones
    listas_filtradas = listas.apply(
        lambda respuestas: [
            opcion
            for opcion in respuestas
            if opcion in opciones_frecuentes
        ]
    )

    # Creamos las columnas binarias
    mlb = MultiLabelBinarizer(classes=opciones_frecuentes)
    matriz = mlb.fit_transform(listas_filtradas)

    # Identificamos el origen de cada opción
    nombres = [
        f"{col}__{opcion}"
        for opcion in mlb.classes_
    ]

    X_multi_parts[col] = pd.DataFrame(
        matriz,
        columns=nombres,
        index=X.index
    )

# Unimos las matrices de todas las columnas multirrespuesta
X_multi = (
    pd.concat(list(X_multi_parts.values()), axis=1)
    if X_multi_parts
    else pd.DataFrame(index=X.index)
)

print("Columnas multirrespuesta originales:", len(multi_cols))
print("Columnas binarias resultantes:", X_multi.shape[1])
print("Columnas nuevas agregadas:", X_multi.shape[1])
print("Umbral de frecuencia:", MIN_FRECUENCIA)

display(X_multi.head())

Columnas multirrespuesta originales: 6
Columnas binarias resultantes: 386
Columnas nuevas agregadas: 386
Umbral de frecuencia: 1


,contas_con_beneficios_adicionales__Horarios Flexibles,contas_con_beneficios_adicionales__Capacitaciones Y/O Cursos,contas_con_beneficios_adicionales__Vacaciones Flexibles (Adicionales A Las Reglamentarias),contas_con_beneficios_adicionales__Abono De Internet,contas_con_beneficios_adicionales__Descuento En Gimnasios / Clases De Gimnasia Online,contas_con_beneficios_adicionales__Clases De Idiomas,contas_con_beneficios_adicionales__Comidas Pagas / Subvencionadas,contas_con_beneficios_adicionales__Licencia Por Nacimiento Extendida,contas_con_beneficios_adicionales__Ninguna De Las Anteriores,contas_con_beneficios_adicionales__Abono De Celular,...,qa_testing__Robot Framework,qa_testing__No Aplica,qa_testing__Zephyr,qa_testing__Karate,qa_testing__K6,qa_testing__Hp Loadrunner,qa_testing__Swagger,qa_testing__Webdriverio,qa_testing__No Uso,qa_testing__Frontier
0,0,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
# Verificamos que todas las columnas de cada grupo tengan la misma cantidad de filas
# Grupos de variables procesadas
grupos = {
    "Numéricas escaladas": {
        "matriz": X_num,
        "variables": num_cols,
        "transformacion": "Estandarización con StandardScaler"
    },
    "Nominales One-Hot": {
        "matriz": X_nom,
        "variables": nom_cols,
        "transformacion": "Codificación One-Hot"
    },
    "Ordinales": {
        "matriz": X_ordinal,
        "variables": ordinal_cols,
        "transformacion": "Codificación ordinal: Junior=0, Semi-Senior=1, Senior=2"
    },
    "Multirrespuesta": {
        "matriz": X_multi,
        "variables": multi_cols,
        "transformacion": (f"Binarización de opciones con más de {MIN_FRECUENCIA} " f"apariciones por variable")
    },
}

# Resumen de las transformaciones aplicadas
print("RESUMEN DEL PREPROCESAMIENTO")
print("=" * 75)

for nombre, info in grupos.items():
    matriz = info["matriz"]
    variables = info["variables"]

    print(f"\n{nombre}")
    print("-" * 75)
    print("Transformación:", info["transformacion"])
    print("Variables originales trabajadas:")
    for variable in variables:
        print(f"  - {variable}")
    print(f"Columnas resultantes: {matriz.shape[1]}")

# Construimos la matriz final incluyendo todos los grupos
X_final = pd.concat(
    [info["matriz"] for info in grupos.values()],
    axis=1
)

# Comprobaciones finales
print("\n" + "=" * 75)
print("MATRIZ FINAL DE PREDICTORES")
print("=" * 75)
print(f"Filas: {X_final.shape[0]}")
print(f"Columnas: {X_final.shape[1]}")
print(f"Dimensiones: {X_final.shape}")

print("\nPrimeras filas de la matriz:")
display(X_final.head(3))


RESUMEN DEL PREPROCESAMIENTO

Numéricas escaladas
---------------------------------------------------------------------------
Transformación: Estandarización con StandardScaler
Variables originales trabajadas:
  - anos_de_experiencia
  - antiguedad_en_la_empresa_actual
  - anos_en_el_puesto_actual
  - tengo_edad
Columnas resultantes: 4

Nominales One-Hot
---------------------------------------------------------------------------
Transformación: Codificación One-Hot
Variables originales trabajadas:
  - donde_estas_trabajando
  - dedicacion
  - tipo_de_contrato
  - modalidad_de_trabajo
  - estas_buscando_trabajo
  - tenes_guardias
  - trabajo_de
  - cantidad_de_personas_en_tu_organizacion
  - maximo_nivel_de_estudios
  - estado
  - carrera
  - genero
Columnas resultantes: 361

Ordinales
---------------------------------------------------------------------------
Transformación: Codificación ordinal: Junior=0, Semi-Senior=1, Senior=2
Variables originales trabajadas:
  - seniority
Columnas 

,anos_de_experiencia,antiguedad_en_la_empresa_actual,anos_en_el_puesto_actual,tengo_edad,donde_estas_trabajando_Buenos Aires,donde_estas_trabajando_Catamarca,donde_estas_trabajando_Chaco,donde_estas_trabajando_Chubut,donde_estas_trabajando_Ciudad Autónoma De Buenos Aires,donde_estas_trabajando_Corrientes,...,qa_testing__Robot Framework,qa_testing__No Aplica,qa_testing__Zephyr,qa_testing__Karate,qa_testing__K6,qa_testing__Hp Loadrunner,qa_testing__Swagger,qa_testing__Webdriverio,qa_testing__No Uso,qa_testing__Frontier
0,0.613263,-0.289608,-0.153412,0.894996,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1.846085,1.419854,2.189779,1.461241,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.372994,-0.669488,0.888007,-0.124246,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [55]:
# Unir X_final con el target 'y' para guardarlos juntos
df_procesado = X_final.copy()
df_procesado["salario"] = y.values

# Guardar en un CSV
df_procesado.to_csv("../data/dataset_features.csv", index=False)
print("¡Dataset procesado guardado con éxito!")

¡Dataset procesado guardado con éxito!
